# P13 without a1, with and without EHUNAM

Trains two P13 variants with seeds 1-5 each (`temporal_model/train.py`):
- `p13_noa1`: every labeled room except a1
- `p13_noa1_ehunam`: the same plus 179 EHUNAM people-counting captures (6 rooms, 0-4 people), converted by `temporal_model/ehunam.py`; half of the windows come from them

Runtime: GPU (Runtime -> Change runtime type -> T4 GPU). About an hour in total, most of it streaming and converting EHUNAM (about 25 GB, nothing is kept but the maps).

Run this notebook in the Colab web UI ([open in Colab](https://colab.research.google.com/github/Agata872/Huawei/blob/main/colab/train_ehunam.ipynb)): it needs one file uploaded and returns a zip, and the upload and download helpers only work there.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

Clone the code (public repository, `main` branch), or update an existing clone.

In [ ]:
import os, subprocess
if not os.path.exists('/content/Huawei'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Agata872/Huawei.git', '/content/Huawei'], check=True)
%cd /content/Huawei
!git pull -q
!git log --oneline -1
!pip install -q 'blosc2>=3,<5' remotezip inflate64 openpyxl

Download the labeled archive (2.1 GB) from the link in `FutureNetwork-starting-kit/data_downloads.md`, check its SHA-256 and extract it.

In [ ]:
%%bash
set -e
mkdir -p data/downloads data/extracted
cd data/downloads
[ -f FutureNetwork-data-labeled.zip ] || wget -q https://dl.r0h.eu/techarena2026-paris/FutureNetwork-data-labeled.zip
echo '1f60df2d4c1459c307e444741056e17a5ec537614cf577331ed04eb94d0765c1  FutureNetwork-data-labeled.zip' | sha256sum -c
unzip -q -n FutureNetwork-data-labeled.zip -d ../extracted
ls ../extracted/labeled | wc -l

Precompute the block maps (CPU, a few minutes) and check them against the raw-input features: the printed difference should be ~1e-6 or smaller.

In [ ]:
!python -m temporal_model.maps --check

Stream the EHUNAM captures from figshare and convert them to block maps (20-40 minutes).
If the cell stops (for example after a disconnect), run it again: converted captures are skipped.

In [ ]:
!python -m temporal_model.ehunam --workers 4

Upload the P11B checkpoint from your local copy of the Reference folder:
`Reference/roomsense_p13_fast/experiments/runs_p11_fast/P11B/checkpoint.pt` (3 MB).
It is not in the repository and only lives in this runtime. Instead of the file picker you can also drag it into `/content/` with the file browser and rename it to `P11B_checkpoint.pt`.

In [ ]:
import os, shutil
P11B = '/content/P11B_checkpoint.pt'
if not os.path.exists(P11B):
    from google.colab import files
    name, = files.upload()
    shutil.move(name, P11B)
!echo 'c6ae4c5814cea75f3522d99ed7e014c7ae2612eb2e49a8cb4a8d7dbb9d42fb82  /content/P11B_checkpoint.pt' | sha256sum -c

Train both variants, seeds 1-5 each (about a minute per seed). Each run prints its loss every 400 steps.

In [ ]:
!python -m temporal_model.train --p11b /content/P11B_checkpoint.pt --seeds 1,2,3,4,5 --exclude-rooms a1 --name p13_noa1
!python -m temporal_model.train --p11b /content/P11B_checkpoint.pt --seeds 1,2,3,4,5 --exclude-rooms a1 --ehunam-share 0.5 --name p13_noa1_ehunam

Download the ten checkpoints and unzip them into the local repository's `runs/temporal_model/`.
Do this before disconnecting: the runtime's files are deleted with it. A copy also goes to Google Drive (`MyDrive/huawei/p13_variants.zip`) if you allow the Drive connection.

In [ ]:
!rm -f /content/p13_variants.zip && zip -qj /content/p13_variants.zip runs/temporal_model/p13_noa1*_seed*.pt && ls -l /content/p13_variants.zip
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/huawei', exist_ok=True)
    shutil.copy('/content/p13_variants.zip', '/content/drive/MyDrive/huawei/p13_variants.zip')
    print('copied to Google Drive: MyDrive/huawei/p13_variants.zip')
except Exception as error:
    print(f'Google Drive unavailable ({error})')
try:
    from google.colab import files
    files.download('/content/p13_variants.zip')
except Exception as error:
    print(f'browser download unavailable ({error}); fetch /content/p13_variants.zip from the file browser')